# 08 · Metal catalysts

Whole catalysts rather than one coordination sphere: `rx.isomers` lists a catalyst's arrangements and ligand stereo
together, and a transition state on two metals is held by its reacting core as
[05_transition_states](05_transition_states.ipynb) holds an organic one. [07_metal](07_metal.ipynb) covers what the
coordination sphere itself can be told to do.

In [ ]:
import rxembed as rx
import xyzrender

## Catalyst isomers

Two square-planar Ni(II) catalysts with an S,N-bound thiosemicarbazide and an isomeric chelate: one binds through a
carboxylate O and an amide N, the other through a carboxylate O and a carbanion C. `rx.isomers` lists each one's
arrangements of its two chelates on the square and both hands of its chelate's stereocentre, one line per species.

In [ ]:
CATALYSTS = {
    "O,N-bound": "C[N]1(C)NC(N)=[S]->[Ni+2]<-12<-[O-]C(=O)C(c1ccccc1)[N-]->2c1ccccc1",
    "O,C-bound": "C[N]1(C)NC(N)=[S]->[Ni+2]<-12<-[O-]C(=O)N(c1ccccc1)[CH-]->2c1ccccc1",
}
for name, smiles in CATALYSTS.items():
    print(name)
    print(rx.isomers(smiles, geometry="SPL").summary())

## Embed every isomer

`rx.embed` takes the whole set and returns an `EnsembleSet`, one `Ensemble` per species, here the O,N-bound catalyst's
four. Each render labels the stereocentre's hand.

In [ ]:
catalysts = rx.embed(rx.isomers(CATALYSTS["O,N-bound"], geometry="SPL"), n=2, seed=1).minimize()
for catalyst in catalysts:
    print(catalyst)
    display(xyzrender.render(xyzrender.load(catalyst.lowest(1).mol), stereo=True))

## A transition state on two metals

In this TS, H2 splits at Mn: one H stays on Mn as a hydride and the other passes as a proton through an ethanol to
the amide N5. `rx.read_xyz` reads both metals from the geometry as one species, the Mn octahedron, the ferrocene and
the ligand stereocentre, as its summary shows. `fix=` holds the reacting core, Mn, N5, the H2 and the ethanol O-H,
where the TS has it, and the rest is embedded anew.

In [ ]:
mn = rx.read_xyz("structures/mn-h2.xyz", charge=-1)
print(rx.isomers(mn).summary())

CORE = [1, 5, 63, 64, 65, 66]  # Mn, N5, the H2 and the ethanol O-H
partial = [(63, 64), (1, 64), (64, 66), (65, 66), (5, 65)]  # the bonds that break and form
ts = rx.embed(mn, fix=CORE, n=4, seed=1).minimize()
print(ts)
for pair in partial:
    print(pair, round(ts.measure(pair)["mean"], 2))
xyzrender.render(xyzrender.load(ts.lowest(1).mol), ts_bonds=[(i + 1, j + 1) for i, j in partial])

## Fac to mer at Mn

The PNN ligand binds Mn fac in the TS. `center="Mn"` enumerates only the Mn arrangements, the ferrocene keeps the
arrangement the geometry shows, and `fix=` keeps the 12 Mn octahedra the held core allows. An arrangement lists the six
donors in trans pairs, so `H63 C61 C62 N5 N6 P2` is mer with P2 trans to N6. Two of the eight mer arrangements, one per
hand, put P2 trans to N6; the other six put N5 trans to P2 or N6 and do not embed. The species has no coordinates, so
`rx.Template` copies the core's from the TS.

In [ ]:
core = rx.Template(mn, {i: i for i in CORE})
allowed = rx.isomers(mn, geometry="OCT", center="Mn", fix=core)
print(allowed.summary())

mer = rx.embed(allowed.select(arrangement="H63 C61 C62 N5 N6 P2"), fix=core, n=4, seed=1).minimize()
print(mer)
xyzrender.render(xyzrender.load(mer.lowest(1).mol), ts_bonds=[(i + 1, j + 1) for i, j in partial])